# Skip edges

`parse_layered()` layers a DAG. An original edge whose endpoints
are more than one layer apart is a **skip**. In `kpnn2` that skip
is not a second kind of layer: it is a column of the target
layer's hop mask, exactly like an edge from the layer directly
below.

<img class="figure-full" src="../figures/skip_memory.svg" alt="Skip memory">

**Figure 1.** A skip is an extra parent of the target, not a
dummy neuron. The source activation is stored and held, then
passed into the target together with the adjacent hop. Solid
arrows are graph edges; dashed arrows are the forward pass.
A, B, and C are a three-node toy; the worked example below
uses a larger graph.

The rest of this page is that same rule as a `LayeredSpec` and
a `MaskedLinear`. It unrolls **one small graph** so each hop
is a named line. The [Getting started](../getting-started/)
notebook uses a loop over `spec.hops` for a graph without
skips; the same loop works here, because skips already sit
inside those masks.


## What a skip edge means

An adjacent chain such as `H1 -> H2` is one hop: a `MaskedLinear`
maps the layers feeding `H2` onto `H2`.

A skip such as `A -> H2` is still a directed edge into `H2`, so
`A` is an extra **parent of the target unit**. The skip crosses
every layer between source and target without touching them: the
source is not copied through dummy units, and `kpnn2` never
inserts identity neurons or generated node names.

![Hop masks](../figures/hop_masks.svg)

**Figure 2.** Every edge entering a layer is in that layer's hop
mask. This is Figure 1 written as hop masks on the graph used
below. `hops[1]` therefore reads layers 0 and 1, and `hops[2]`
reads layers 0, 1 and 2. Solid and dashed edges are handled by the
same matrix multiply; the dashes only mark which edges jump a
layer.

Three properties follow, and they are the reason the design looks
like this:

- **No edge can be dropped silently.** Applying a hop applies
  every parent of its target at once. There is no second call to
  remember, and `gather_hop_inputs()` raises `Kpnn2Error` if a
  layer that a hop reads was never stored.
- **The fan-in is right.** `MaskedLinear` initializes each output
  row from that row's mask degree. Because skip parents sit in the
  same row, a unit with two adjacent and three skip parents is
  initialized with fan-in five, not two.
- **A skip weight is an ordinary weight.** One weight per incoming
  edge, all in one matrix, all drawn from the same degree-aware
  initialization. There is no skip bias: one bias per unit stays
  on `MaskedLinear`.

In symbols, for target `H2` with adjacent parent `H1` and skip
parent `A`:

```text
H2 = relu( w_{H1→H2} H1 + w_{A→H2} A + b_{H2} )
```

`map_node_attributions()` labels original `layer_nodes` names, so
if a skip affects the prediction, that effect appears on the named
target unit and never on a dummy channel.


## The graph in Figure 2

Inputs `A` and `B`, hidden units `H1` and `H2`, output `C`.
Adjacent edges are `A -> H1`, `B -> H1`, `H1 -> H2`, and
`H2 -> C`. Skips are `A -> H2`, `H1 -> C`, and `A -> C`.


In [1]:
import pandas as pd

import kpnn2 as k2

edgelist = pd.DataFrame(
    {
        "source": ["A", "B", "H1", "H2", "A", "H1", "A"],
        "target": ["H1", "H1", "H2", "C", "H2", "C", "C"],
    }
)
spec = k2.parse_layered(edgelist)
print(spec.layer_nodes)
print([f"{s.source} -> {s.target}" for s in spec.skips])

(('A', 'B'), ('H1',), ('H2',), ('C',))
['A -> H2', 'H1 -> C', 'A -> C']


`spec.hops` has one mask per layer after the inputs. Each row is
the target unit; each column is a parent, including skips. A `0`
is an absent edge (`B` never feeds `H2` or `C`).


In [2]:
hop0, hop1, hop2 = spec.hops
print(
    pd.DataFrame(
        hop0.mask.numpy(), columns=list(hop0.source_nodes), index=["H1"]
    )
)
print()
print(
    pd.DataFrame(
        hop1.mask.numpy(), columns=list(hop1.source_nodes), index=["H2"]
    )
)
print()
print(
    pd.DataFrame(
        hop2.mask.numpy(), columns=list(hop2.source_nodes), index=["C"]
    )
)
ones = int(hop0.mask.sum() + hop1.mask.sum() + hop2.mask.sum())
print(ones, "ones in the masks,", len(edgelist), "edges")

      A    B
H1  1.0  1.0

      A    B   H1
H2  1.0  0.0  1.0

     A    B   H1   H2
C  1.0  0.0  1.0  1.0
7 ones in the masks, 7 edges


`hops[0]` reads only `(A, B)`, because `H1` can only have
layer-0 parents. `hops[1]` also reads `A`, since `A -> H2` jumps
a layer. `hops[2]` reads `A`, `H1`, and `H2`.

The ones across all three masks add up to the number of rows in
the edgelist. That equality is the guarantee: every prior-knowledge
edge is in exactly one mask.


## Three named layers

A normal `nn.Module`: `__init__` builds one `MaskedLinear` per
hop, `forward` applies them in order. The hop into `H1` reads
only the input, so it takes `x` directly. The hops into `H2` and
`C` also need earlier activations, so `gather_hop_inputs()`
concatenates those layers into one tensor whose columns match the
mask.

No skip object is applied anywhere. Bias is off so the numbers
below stay readable.


In [3]:
import torch
import torch.nn.functional as F
from torch import nn


class Net(nn.Module):
    def __init__(self, spec: k2.LayeredSpec):
        super().__init__()
        self.spec = spec
        self.to_h1 = k2.MaskedLinear(spec.hops[0].mask, bias=False)
        self.to_h2 = k2.MaskedLinear(spec.hops[1].mask, bias=False)
        self.to_c = k2.MaskedLinear(spec.hops[2].mask, bias=False)

    def forward(self, x):
        h1 = F.relu(self.to_h1(x))
        h2_in = k2.gather_hop_inputs({0: x, 1: h1}, self.spec.hops[1])
        h2 = F.relu(self.to_h2(h2_in))
        c_in = k2.gather_hop_inputs({0: x, 1: h1, 2: h2}, self.spec.hops[2])
        self.h1, self.h2 = h1, h2
        return self.to_c(c_in)


model = Net(spec)

### Numerical check

`forward` returns `C`. `self.h1` and `self.h2` are stored only so
this check can print the hidden units.

Pin every adjacent weight to `1`, and the skip weights to
`w_{A→H2} = 0.3`, `w_{H1→C} = 0.2`, `w_{A→C} = 0.1`. Columns
follow `source_nodes` above, so absent parents stay `0`.

For input `A = 2`, `B = 0`:

```text
H1 = relu(A + B) = 2
H2 = relu(0.3 A + H1) = relu(2.6) = 2.6
C  = 0.1 A + 0.2 H1 + H2 = 3.2
```

For `A = -1`, `B = 0`, ReLU zeros the path through `H1` and `H2`,
but the skip `A -> C` still contributes:

```text
H1 = relu(-1) = 0
H2 = relu(0.3 (-1) + 0) = 0
C  = 0.1 (-1) + 0.2 * 0 + 0 = -0.1
```

Assigning `layer.weight = pinned` writes through the mask
parametrization into the trainable tensor; the mask then hides
whatever the pinned matrix says about absent edges.


In [4]:
model.to_h1.weight = torch.tensor([[1.0, 1.0]])
model.to_h2.weight = torch.tensor([[0.3, 0.0, 1.0]])
model.to_c.weight = torch.tensor([[0.1, 0.0, 0.2, 1.0]])

c = model(torch.tensor([[2.0, 0.0]]))
print(
    f"A=2  H1, H2, C: {model.h1.item():.1f},"
    f" {model.h2.item():.1f}, {c.item():.1f}"
)
c = model(torch.tensor([[-1.0, 0.0]]))
print(
    f"A=-1 H1, H2, C: {model.h1.item():.1f},"
    f" {model.h2.item():.1f}, {c.item():.1f}"
)

A=2  H1, H2, C: 2.0, 2.6, 3.2
A=-1 H1, H2, C: 0.0, 0.0, -0.1


In a larger graph you would loop the same three steps
(`saved = {0: x}`, then for each hop: gather, `MaskedLinear`,
optional ReLU, `saved[hop.target_layer] = hidden`). Getting
started shows that loop. Nothing extra is added for skips.

## Two things worth checking

The fan-in that `MaskedLinear` initializes from is the row degree
of the hop mask, so it counts skip parents. And a hop that reads a
layer you never stored is an error, not a silent omission.


In [5]:
print("H1 fan-in", int(spec.hops[0].mask.sum()))
print("H2 fan-in", int(spec.hops[1].mask.sum()))
print("C fan-in", int(spec.hops[2].mask.sum()))

x = torch.tensor([[2.0, 0.0]])
try:
    k2.gather_hop_inputs({0: x}, spec.hops[2])
except k2.Kpnn2Error as error:
    print(error)

H1 fan-in 2
H2 fan-in 2
C fan-in 3
saved is missing layer 1. The hop into layer 3 reads layers [0, 1, 2].
